# Setup

In [19]:
# Run ONLY once. Working directory should be 02-machine-translation
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Working directory:", Path.cwd())

Working directory: /Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/31-projects-2026


In [20]:
# Load autoreload extension for Jupyter Notebook
%load_ext autoreload
%autoreload 2

# Import Config, Dataset, and other necessary modules
from src.machine_translation.config import TatoebaConfig
from src.machine_translation.dataset import TatoebaData
from src.machine_translation.model import TatoebaModelPackedSeq
from src.machine_translation.experiment import ExperimentRunnerWithCustomLogging

# Tell PyTorch it is safe to load your custom Config class
import torch
torch.serialization.add_safe_globals([TatoebaConfig, TatoebaData])

# Set up logging format and level
import logging
# logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logging.basicConfig(format="%(levelname)s:%(name)s:  %(message)s")

# Set Pytorch Lightning logging level to WARNING to reduce verbosity
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
logging.getLogger("lightning_fabric").setLevel(logging.WARNING)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# Set up a logger for "tatoeba" level to DEBUG for this notebook
logger = logging.getLogger("tatoeba")
logger.setLevel(logging.DEBUG)

In [10]:
# Set up a logger for "tatoeba" level to INFO for this notebook
logger = logging.getLogger("tatoeba")
logger.setLevel(logging.INFO)

# 1 The First Run on a Toy Dataset

## 01 Downgraded model for a single epoch

In [4]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Downgrade some parameters for a quick test run
config.hidden_dim = 128
config.embedding_dim = 128
config.epochs = 1

In [9]:
# Create an instance of the ExperimentRunnerWithCustomLogging class
runner = ExperimentRunnerWithCustomLogging(
    config=config,
    run_name="base_config_data-limit=1000_hidden=128_emb=128_epochs=1",
    data_limit=1000,  # Limit the number of data samples for quick testing
)

DEBUG:tatoeba.trainer:  ===TRAINER INITIALIZATION===
DEBUG:tatoeba.trainer:  Device set to: auto
DEBUG:tatoeba.trainer:  Random seed in Lightning set to: 42
DEBUG:tatoeba.trainer:  ===CONFIG INITIALIZATION===
DEBUG:tatoeba.trainer:  Using provided config instance. Run name: base_config_data-limit=1000_hidden=128_emb=128_epochs=1
DEBUG:tatoeba.dataset:  === DATASET CREATION ===
DEBUG:tatoeba.dataset:  === prepare_data() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/tatoeba
DEBUG:tatoeba.dataset:  Loaded ONLY train data: 800
DEBUG:tatoeba.dataset:  === _train_and_save_tokenizer() call ===
DEBUG:tatoeba.dataset:  Tokenizer already exists at datasets/tokenizers/tokenizer_limit_1000.json. Skipping training.
DEBUG:tatoeba.dataset:  === setup() call ===
DEBUG:tatoeba.dataset:  === _load_and_split_data() call ===
DEBUG:tatoeba.dataset:  Loading Tatoeba dataset from local disk cache: datasets/

In [11]:
runner.fit()

🚀 Using hardware accelerator: mps:0


/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve per

Epoch   1 | Train Loss: 6.8694 | Train Acc: 0.1988 | Val Loss: 6.9066 | Val Acc: 0.1793

✅ Training finished. Extract best model metrics.

🏆 Best Model Metrics (from Epoch 1):
├─ Train Loss: 6.8694
├─ Train Acc:  0.1988
├─ Val Loss:   6.9066
└─ Val Acc:    0.1793


## 02 Downgraded model for some more epochs

In [16]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Downgrade some parameters for a quick test run
config.hidden_dim = 128
config.embedding_dim = 128
config.epochs = 50
config.patience = 50  # Set patience for early stopping

# Create an instance of the ExperimentRunnerWithCustomLogging class
runner = ExperimentRunnerWithCustomLogging(
    config=config,
    run_name="base_config_data-limit=1000_hidden=128_emb=128_epochs=50_patience=50",
    data_limit=1000,  # Limit the number of data samples for quick testing
    print_every_n_epochs=5,  # Print metrics every 5 epochs
)

# Fit the model using the runner
runner.fit()

🚀 Using hardware accelerator: mps:0


/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve per

Epoch   5 | Train Loss: 3.2705 | Train Acc: 0.4462 | Val Loss: 7.0472 | Val Acc: 0.1991
Epoch  10 | Train Loss: 2.2623 | Train Acc: 0.5956 | Val Loss: 7.2364 | Val Acc: 0.1879
Epoch  15 | Train Loss: 2.0705 | Train Acc: 0.6351 | Val Loss: 7.2854 | Val Acc: 0.1875
Epoch  20 | Train Loss: 2.0396 | Train Acc: 0.6412 | Val Loss: 7.2937 | Val Acc: 0.1875
Epoch  25 | Train Loss: 2.0310 | Train Acc: 0.6417 | Val Loss: 7.2958 | Val Acc: 0.1875
Epoch  30 | Train Loss: 2.0289 | Train Acc: 0.6423 | Val Loss: 7.2971 | Val Acc: 0.1875
Epoch  35 | Train Loss: 2.0264 | Train Acc: 0.6426 | Val Loss: 7.2981 | Val Acc: 0.1875
Epoch  40 | Train Loss: 2.0220 | Train Acc: 0.6432 | Val Loss: 7.2994 | Val Acc: 0.1875
Epoch  45 | Train Loss: 2.0162 | Train Acc: 0.6438 | Val Loss: 7.3007 | Val Acc: 0.1875
Epoch  50 | Train Loss: 2.0139 | Train Acc: 0.6441 | Val Loss: 7.3018 | Val Acc: 0.1879

✅ Training finished. Extract best model metrics.

🏆 Best Model Metrics (from Epoch 2):
├─ Train Loss: 5.1979
├─ Train A

## 03 Less Downgraded model for some more epochs

In [17]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Downgrade some parameters for a quick test run
config.hidden_dim = 256
config.embedding_dim = 256
config.epochs = 10
config.patience = 10  # Set patience for early stopping

# Create an instance of the ExperimentRunnerWithCustomLogging class
runner = ExperimentRunnerWithCustomLogging(
    config=config,
    run_name="base_config_data-limit=1000_hidden=256_emb=256_epochs=10_patience=10",
    data_limit=1000,  # Limit the number of data samples for quick testing
    print_every_n_epochs=5,  # Print metrics every 5 epochs
)

# Fit the model using the runner
runner.fit()

🚀 Using hardware accelerator: mps:0


/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve per

Epoch   5 | Train Loss: 1.5359 | Train Acc: 0.7322 | Val Loss: 7.3826 | Val Acc: 0.1909
Epoch  10 | Train Loss: 0.5058 | Train Acc: 0.9700 | Val Loss: 7.5754 | Val Acc: 0.1922

✅ Training finished. Extract best model metrics.

🏆 Best Model Metrics (from Epoch 1):
├─ Train Loss: 6.4342
├─ Train Acc:  0.2194
├─ Val Loss:   6.8643
└─ Val Acc:    0.1879


## 04 Run with a BLUE score metric

### 01 BLEU score after each epoch

In [18]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Downgrade some parameters for a quick test run
config.hidden_dim = 256
config.embedding_dim = 256
config.epochs = 10
config.patience = 10  # Set patience for early stopping

# Create an instance of the ExperimentRunnerWithCustomLogging class
runner = ExperimentRunnerWithCustomLogging(
    config=config,
    run_name="base_config_data-limit=1000_hidden=256_emb=256_epochs=10_patience=10",
    data_limit=1000,  # Limit the number of data samples for quick testing
    print_every_n_epochs=5,  # Print metrics every 5 epochs
)

# Fit the model using the runner
runner.fit()

🚀 Using hardware accelerator: mps:0


/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/31-projects-2026/02-machine-translation/checkpoints/base_config_data-limit=1000_hidden=256_emb=256_epochs=10_patience=10 exists and is not empty.
/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7

Epoch   5 | Train Loss: 1.5359 | Train Acc: 0.7322 | Val Loss: 7.3826 | Val Acc: 0.1909 | Val BLEU: 0.0007
Epoch  10 | Train Loss: 0.5058 | Train Acc: 0.9700 | Val Loss: 7.5754 | Val Acc: 0.1922 | Val BLEU: 0.0009

✅ Training finished. Extract best model metrics.

🏆 Best Model Metrics (from Epoch 1):
├─ Train Loss: 6.4342
├─ Train Acc:  0.2194
├─ Val Loss:   6.8643
├─ Val Acc:    0.1879
└─ Val BLEU:   0.0000


### 02 BLEU score only at the end of training

In [25]:
# Create a congig object for Tatoeba dataset
config = TatoebaConfig()

# Downgrade some parameters for a quick test run
config.hidden_dim = 256
config.embedding_dim = 256
config.epochs = 10
config.patience = 10  # Set patience for early stopping

# Create an instance of the ExperimentRunnerWithCustomLogging class
runner = ExperimentRunnerWithCustomLogging(
    config=config,
    run_name="base_config_data-limit=1000_hidden=256_emb=256_epochs=10_patience=10_bleu-end-only",
    data_limit=1000,  # Limit the number of data samples for quick testing
    print_every_n_epochs=1,  # Print metrics every 1 epoch
)

# Fit the model using the runner
runner.fit()

🚀 Using hardware accelerator: mps:0


/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/31-projects-2026/checkpoints/base_config_data-limit=1000_hidden=256_emb=256_epochs=10_patience=10_bleu-end-only exists and is not empty.
/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the 

Epoch   1 | Train Loss: 6.4342 | Train Acc: 0.2194 | Val Loss: 6.8643 | Val Acc: 0.1879
Epoch   2 | Train Loss: 4.5369 | Train Acc: 0.3108 | Val Loss: 7.0903 | Val Acc: 0.2021
Epoch   3 | Train Loss: 3.1833 | Train Acc: 0.4311 | Val Loss: 7.1977 | Val Acc: 0.1969
Epoch   4 | Train Loss: 2.1008 | Train Acc: 0.6064 | Val Loss: 7.2952 | Val Acc: 0.1999
Epoch   5 | Train Loss: 1.5359 | Train Acc: 0.7322 | Val Loss: 7.3826 | Val Acc: 0.1909
Epoch   6 | Train Loss: 1.0833 | Train Acc: 0.8466 | Val Loss: 7.4485 | Val Acc: 0.1909
Epoch   7 | Train Loss: 0.8595 | Train Acc: 0.8998 | Val Loss: 7.4861 | Val Acc: 0.1905
Epoch   8 | Train Loss: 0.6737 | Train Acc: 0.9395 | Val Loss: 7.5232 | Val Acc: 0.1918
Epoch   9 | Train Loss: 0.5853 | Train Acc: 0.9560 | Val Loss: 7.5553 | Val Acc: 0.1900
Epoch  10 | Train Loss: 0.5058 | Train Acc: 0.9700 | Val Loss: 7.5754 | Val Acc: 0.1922

🏆 Computing BLEU score for the best model...


/Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/env/lib/python3.14/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.



✅ Training finished. Extract best model metrics.

🏆 Best Model Metrics (from Epoch 1):
├─ Train Loss: 6.4342
├─ Train Acc:  0.2194
├─ Val Loss:   6.8643
├─ Val Acc:    0.1879
└─ Val BLEU:   0.0000

📂 Loading best model from checkpoint: /Users/iliarudiak/Library/Mobile Documents/com~apple~CloudDocs/_working/2026/31-projects-2026/checkpoints/base_config_data-limit=1000_hidden=256_emb=256_epochs=10_patience=10_bleu-end-only/best-model-v2.ckpt
